In [1]:
from transformers import AutoTokenizer

- 加载分词器
- API 调用
  - `tokenizer.tokenize(text)`
  - `tokenizer.convert_tokens_to_ids(tokens)`
  - `tokenizer.convert_ids_to_tokens(ids)`
  - `tokenizer.encode(text, ...)`
  - `tokenizer.decode(ids, ...)`
  - `tokenizer(...)` = batch_encode
- 分词器与模型配合使用
  - 原始模型
  - 带任务头的模型

# 加载分词器

In [3]:
# 从远程加载
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-chinese")
print(tokenizer)

BertTokenizer(name_or_path='google-bert/bert-base-chinese', vocab_size=21128, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [4]:
# 从本地加载
PATH = 'D:/AI/hugging face/bert-base-chinese'
tokenizer = AutoTokenizer.from_pretrained(PATH)
print(tokenizer)

BertTokenizer(name_or_path='D:/AI/hugging face/bert-base-chinese', vocab_size=21128, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


# API 调用

In [5]:
text = '我爱自然语言处理'

In [6]:
# 1. 分词
tokens = tokenizer.tokenize(text)
print(tokens)

['我', '爱', '自', '然', '语', '言', '处', '理']


In [ ]:
# 2. token -> id
ids = tokenizer.encode(text, add_special_tokens=True)
print(ids)
# 查看 vocab.txt 可以知道，101 为 <CLS>，102 为 <SEP>

[101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415, 102]


In [8]:
# 3. id -> token
tokens = tokenizer.convert_ids_to_tokens(ids)
print(tokens)

['[CLS]', '我', '爱', '自', '然', '语', '言', '处', '理', '[SEP]']


In [9]:
# 4. encoding (text -> ids)
ids = tokenizer.encode(text, add_special_tokens=True)
print(ids)

[101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415, 102]


In [ ]:
ids = tokenizer.encode(text,
                       text_pair='你爱打篮球吗？',
                       add_special_tokens=True,
                       padding='max_length',
                       max_length=20,
                       truncation=True
                       )
print(ids)

[101, 2769, 4263, 5632, 4197, 6427, 6241, 1905, 4415, 102, 872, 4263, 2802, 5074, 4413, 1408, 8043, 102, 0, 0]


In [12]:
# 5. decoding (ids -> text)
sentence = tokenizer.decode(ids, skip_special_tokens=True)
print(sentence)

我 爱 自 然 语 言 处 理 你 爱 打 篮 球 吗 ？


In [22]:
# 6. __call__ 方法（加强版批量编码）
texts = ['喜欢游泳', '不吃香菜', '讨厌下雨天']
texts_pairs = ['喜欢', '讨厌', '讨厌']

In [23]:
inputs = tokenizer(
    texts,
    text_pair=texts_pairs,
    add_special_tokens=True,
    padding=True,   # longest
    return_tensors='pt'
)
print([k for k, v in inputs.items()])
print(inputs['input_ids'])
print(inputs['token_type_ids'])
print(inputs['attention_mask'])

['input_ids', 'token_type_ids', 'attention_mask']
tensor([[ 101, 1599, 3614, 3952, 3807,  102, 1599, 3614,  102,    0],
        [ 101,  679, 1391, 7676, 5831,  102, 6374, 1328,  102,    0],
        [ 101, 6374, 1328,  678, 7433, 1921,  102, 6374, 1328,  102]])
tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


# 分词器配合模型

In [24]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

## 不带任务头的模型

In [25]:
# 1. 准备数据
texts = ['喜欢游泳', '不吃香菜', '讨厌下雨天']

In [26]:
# 2. 加载模型与分词器
MODEL_NAME_OR_PATH = 'D:/AI/hugging face/bert-base-chinese'
model = AutoModel.from_pretrained(MODEL_NAME_OR_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: D:/AI/hugging face/bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# 3. 分词器编码文本，获得模型输入
inputs = tokenizer(
    texts,
    add_special_tokens=True,
    padding=True,   # longest
    return_tensors='pt'
)
print(inputs)

{'input_ids': tensor([[ 101, 1599, 3614, 3952, 3807,  102,    0],
        [ 101,  679, 1391, 7676, 5831,  102,    0],
        [ 101, 6374, 1328,  678, 7433, 1921,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1]])}


In [32]:
# 4. 模型的前向传播
outputs = model(
    input_ids=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    token_type_ids=inputs['token_type_ids']
)

# 可用 字典解包 操作简化
outputs = model(**inputs)
print(outputs.keys())
print(outputs.last_hidden_state.shape)  # (N, L, D)
print(outputs.pooler_output.shape)    # (N, D)

odict_keys(['last_hidden_state', 'pooler_output'])
torch.Size([3, 7, 768])
torch.Size([3, 768])


## 带任务头的模型

In [41]:
import torch

# 1. 准备数据
texts = ['喜欢游泳', '不吃香菜', '讨厌下雨天']
# 2. 加载模型与分词器
MODEL_NAME_OR_PATH = 'D:/AI/hugging face/bert-base-chinese'
model4sc = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_OR_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: D:/AI/hugging face/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
inputs = tokenizer(
    texts,
    padding=True,
    return_tensors='pt'
)
print(inputs)

{'input_ids': tensor([[ 101, 1599, 3614, 3952, 3807,  102,    0],
        [ 101,  679, 1391, 7676, 5831,  102,    0],
        [ 101, 6374, 1328,  678, 7433, 1921,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1]])}


In [43]:
import torch
labels = torch.randint(2, (3,))
inputs['labels'] = labels   # 分类标签（二分类）
print(inputs)

{'input_ids': tensor([[ 101, 1599, 3614, 3952, 3807,  102,    0],
        [ 101,  679, 1391, 7676, 5831,  102,    0],
        [ 101, 6374, 1328,  678, 7433, 1921,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([0, 1, 1])}


In [45]:
outputs = model4sc(**inputs)
print(outputs)
print(outputs.logits)
print(outputs.loss)

SequenceClassifierOutput(loss=tensor(0.6468, grad_fn=<NllLossBackward0>), logits=tensor([[-0.4307,  0.8439],
        [-0.1922,  1.2090],
        [-0.4827,  1.0307]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[-0.4307,  0.8439],
        [-0.1922,  1.2090],
        [-0.4827,  1.0307]], grad_fn=<AddmmBackward0>)
tensor(0.6468, grad_fn=<NllLossBackward0>)


In [46]:
outputs.loss.backward()